In [27]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
import tqdm
from torch.utils.tensorboard import SummaryWriter

In [3]:
df_train = pd.read_csv(('../data/train_multilabel.csv'))
df_val = pd.read_csv(('../data/test_multilabel.csv'))
df_test = pd.read_csv(('../data/valid_multilabel.csv'))


In [13]:
label_cols = df_train.columns[6:].tolist()

In [ ]:
#데이터 셋 만들기(멀티라벨 용으로)

class ComplainDataset(Dataset):
    def __init__(self, df, tokenizer, label_cols, max_length=128):
        self.df = df
        self.tokenizer = tokenizer
        self.label_cols = label_cols
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        text = self.df.loc[idx,'text']
        
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )

        label = self.df.loc[idx, self.label_cols].values.astype("float32")

        item = {
            "input_ids": encoding['input_ids'].squeeze(0),
            "attention_mask": encoding['attention_mask'].squeeze(0),
            "labels": torch.tensor(label, dtype=torch.float32) 
        }
        return item

In [12]:
# 한국어 전용 토크나이져
tokenizer = AutoTokenizer.from_pretrained("klue/roberta-base")

c:\Source\TEAM-PJ-DEEP\.venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\dustk\.cache\huggingface\hub\models--klue--roberta-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [19]:
#dataset 만들기
train_dataset = ComplainDataset(df_train, tokenizer, label_cols)
val_dataset = ComplainDataset(df_val, tokenizer, label_cols)
test_dataset = ComplainDataset(df_test, tokenizer, label_cols)

In [21]:
#data_loaders 만들기
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)

In [38]:
# 모델 생성
model = AutoModelForSequenceClassification.from_pretrained(
    "klue/roberta-base",
    num_labels=len(label_cols),
    problem_type="multi_label_classification"
)

for params in model.parameters():
    params.requires_grad = False

for param in model.classifier.parameters():
    param.requires_grad = True

for param in model.roberta.encoder.layer[11].parameters():
    param.requires_grad = True

for name,module in model.named_parameters():
    print(name, module.requires_grad)

model

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 942.71it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]              
RobertaForSequenceClassification LOAD REPORT from: klue/roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Co

roberta.embeddings.word_embeddings.weight False
roberta.embeddings.token_type_embeddings.weight False
roberta.embeddings.LayerNorm.weight False
roberta.embeddings.LayerNorm.bias False
roberta.embeddings.position_embeddings.weight False
roberta.encoder.layer.0.attention.self.query.weight False
roberta.encoder.layer.0.attention.self.query.bias False
roberta.encoder.layer.0.attention.self.key.weight False
roberta.encoder.layer.0.attention.self.key.bias False
roberta.encoder.layer.0.attention.self.value.weight False
roberta.encoder.layer.0.attention.self.value.bias False
roberta.encoder.layer.0.attention.output.dense.weight False
roberta.encoder.layer.0.attention.output.dense.bias False
roberta.encoder.layer.0.attention.output.LayerNorm.weight False
roberta.encoder.layer.0.attention.output.LayerNorm.bias False
roberta.encoder.layer.0.intermediate.dense.weight False
roberta.encoder.layer.0.intermediate.dense.bias False
roberta.encoder.layer.0.output.dense.weight False
roberta.encoder.layer.

RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(32000, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
             

In [41]:
writer = SummaryWriter()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-5)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
criterion = nn.BCEWithLogitsLoss()

model.to(device)

best_val_loss = 100000000
epochs = 20
early_stop_epochs = 3
early_stop_counter = 0
count = 0

for epoch in range(epochs):
    train_tqdm = tqdm.tqdm(train_loader)
    model.train()
    train_loss_sum = 0

    for batch in train_tqdm:
        optimizer.zero_grad()

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits

        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        
        train_loss_sum += loss.item()

        writer.add_scalar("Loss/train_step", loss.item(), count)
        count += 1

        train_tqdm.set_postfix(loss=f"{loss.item():.4f}")

    avg_train_loss = train_loss_sum / len(train_loader)
    print("avg_train_loss",avg_train_loss)

    model.eval()    
    all_preds = []
    all_labels = []
    val_loss_sum = 0
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits

            loss = criterion(logits, labels)
            val_loss_sum += loss.item()    

            probs = torch.sigmoid(logits)
            preds = (probs > 0.5).float()

            all_preds.append(preds.cpu())
            all_labels.append(labels.cpu())

        avg_val_loss = val_loss_sum / len(val_loader)
    print( "avg_val_loss",avg_val_loss)
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), "multiLabel_best_model.pt")
        early_stop_counter = 0
    else:
        early_stop_counter += 1

        if early_stop_counter >= early_stop_epochs:
            print("Early stopping triggered.")
            break

    

    



100%|██████████| 500/500 [00:23<00:00, 20.88it/s, loss=0.0833]


avg_train_loss 0.08967317366600036
avg_val_loss 0.06437559017709865


100%|██████████| 500/500 [00:23<00:00, 21.15it/s, loss=0.0663]


avg_train_loss 0.07426377455145121
avg_val_loss 0.05698463273276189


100%|██████████| 500/500 [00:23<00:00, 21.18it/s, loss=0.0546]


...

avg_train_loss 0.015099252434447407
avg_val_loss 0.04357324987887197


100%|██████████| 500/500 [00:23<00:00, 21.25it/s, loss=0.0108]


avg_train_loss 0.012740546209737658
avg_val_loss 0.046269310982364


100%|██████████| 500/500 [00:23<00:00, 21.33it/s, loss=0.0092]


avg_train_loss 0.01079466557689011
avg_val_loss 0.04734404436104996
Early stopping triggered.


In [42]:
model.eval()

test_loss_sum = 0
all_preds = []
all_labels = []
test_tqdm = tqdm.tqdm(test_loader, desc="Test")
with torch.no_grad():
    for batch in test_tqdm:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        loss = criterion(logits, labels)
        test_loss_sum += loss.item()
        probs = torch.sigmoid(logits)
        preds = (probs >= 0.5).float()
        all_preds.append(preds.cpu())
        all_labels.append(labels.cpu())
        test_tqdm.set_postfix(loss=f"{loss.item():.4f}")
avg_test_loss = test_loss_sum / len(test_loader)

avg_test_loss

Test: 100%|██████████| 125/125 [00:05<00:00, 24.68it/s, loss=0.0087]


0.008330016270279884

In [53]:
text = "여권 만들고싶어"

model.eval()

encoding = tokenizer(
    text,
    truncation=True,
    padding="max_length",
    max_length=128,
    return_tensors="pt"
)
input_ids = encoding["input_ids"].to(device)
attention_mask = encoding["attention_mask"].to(device)
with torch.no_grad():
    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
    logits = outputs.logits
    probs = torch.sigmoid(logits).squeeze(0).cpu().numpy()
pred_labels = []
for col, prob in zip(label_cols, probs):
    if prob >= 0.3:
        pred_labels.append((col, float(prob)))
pred_labels = sorted(pred_labels, key=lambda x: x[1], reverse=True)

print(pred_labels)
print(probs)

[('문서:신분증', 0.9877126812934875), ('기관:시·군·구청', 0.9791040420532227), ('부서:민원여권과/여권팀', 0.9703047871589661), ('문서:여권신청서', 0.812505304813385), ('문서:증명사진', 0.7602254152297974), ('문서:수수료', 0.7538648843765259), ('민원명:여권 발급', 0.3480336666107178)]
[1.5999090e-03 2.6117091e-04 7.4779051e-03 3.5571354e-03 6.9164630e-04
 4.2428434e-04 6.4482703e-04 2.8875249e-03 9.7910404e-01 1.1308747e-03
 2.8264821e-03 5.0563239e-03 8.9551955e-03 3.3549436e-03 2.2754951e-03
 1.4251154e-03 7.5125610e-03 2.6532323e-03 4.7336108e-04 2.4659364e-03
 5.8723899e-04 7.3549961e-04 1.0180727e-03 1.4365162e-03 1.2355933e-01
 2.1533556e-03 8.3114713e-04 4.8809205e-04 3.6865426e-03 3.1850424e-03
 6.8090402e-04 5.3158059e-04 1.8511025e-03 6.5370498e-04 1.8207160e-03
 1.7972803e-04 2.0633121e-01 4.3124943e-03 1.2189973e-04 2.5828197e-03
 7.8252103e-04 1.5019358e-04 8.5656391e-04 1.5529416e-03 4.3063509e-04
 4.6298208e-04 3.5469883e-04 6.1431824e-04 1.2182472e-03 2.7818957e-03
 1.4225466e-03 7.4461591e-04 1.0033774e-01 5.249502